In [ ]:
# !pip install pandas

In [ ]:
# !pip install pdfplumber 

In [1]:
import requests
OLLAMA_URL = "http://localhost:11434"
GEN_MODEL = "llama3:latest"  
EMBED_MODEL = "llama3:latest"  

In [2]:
# sanity check- ollama is upto date and avaliable
def check_ollama():
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models",[])]
        print("ollama is up-to-date")
        for m in models:
            print(f" -{m}")
        for required in (GEN_MODEL, EMBED_MODEL):
            if not any(required in m for m in models):
                print(f"\n[!] '{required}' not found. Run: ollama pull {required}")
        return models, "model version"
    except Exception as e:
        print(f"[!] cannot reach ollama at {OLLAMA_URL}:{e}")
        print("start it with 'ollama serve' in a separate terminal.")
    
check_ollama()

ollama is up-to-date
 -llama3:latest


(['llama3:latest'], 'model version')

importing raw text from pdf 

In [3]:
import pdfplumber

PDF_PATH = "C:\\Users\\Mateti sai pranay\\Downloads\\.net\\python\\re_vamp\\pdf-text_pipline\\docs_images\\invoices\\medical_similar\\shiva\\invoice1.pdf"  # <-- change this to your PDF file path

text = ""
tables = []

with pdfplumber.open(PDF_PATH) as pdf:
    for page in pdf.pages:
        text += page.extract_text() or ""
        for t in page.extract_tables():
            if t:
                tables.append({"page": page.page_number, "rows": t})

print(f"Extracted {len(text)} chars of text and {len(tables)} tables.")
print("\n--- TEXT PREVIEW ---")
print(text[:500])

Extracted 2542 chars of text and 1 tables.

--- TEXT PREVIEW ---
Party Name :
SRI SHIVA SAI POULTRY & VETERINARY NEEDS
BHARGAV MEDICAL STORES
GST INVOICE
STALL NO 14 BUS STAND COMPLEX KOTHAGUDEM
D.NO.7-146/1, ROOM NO-1,KOTHAPETA ROAD,
BHADRADRI KOTHAGUDEM DIST - 507101. CREDIT DAMMAPETA(V),DAMMAPETA(M),BHADRADRI(D),TG,IND
Invoice No B001000786 Order No. Cases 0 36-TELANGANA
Phone : 9848091819 Order Date PHONE. : 9866514042
D.L.No. : TG/22/02/2001-20590,20591,20592,593 Invoice Date 13-07-2023 L.R. No. Transport D.L.No. : TS/KGM/2023-106432
GSTIN : 36AAOFS3456N


In [4]:
# inspecting the tables pdfplumber 
for i, t in enumerate(tables):
    print(f"\n == table {i+1} (page {t['page']}, {len(t['rows'])} rows)===")
    for row in t["rows"][:5]:
        print(row)
    if len(t["rows"]) > 5:
        print(f"... +{len(t['rows'])-5} more rows")
    pass


 == table 1 (page 1, 12 rows)===
['SRI SHIVA SAI POULTRY & VETERINARY NEEDS\nSTALL NO 14 BUS STAND COMPLEX KOTHAGUDEM\nBHADRADRI KOTHAGUDEM DIST - 507101.\nPhone : 9848091819\nD.L.No. : TG/22/02/2001-20590,20591,20592,593\nGSTIN : 36AAOFS3456N1ZG', None, None, None, None, None, None, 'GST INVOICE\nCREDIT', None, None, None, None, None, None, None, None, None, None, None, None, None, 'Party Name :\nBHARGAV MEDICAL STORES\nD.NO.7-146/1, ROOM NO-1,KOTHAPETA ROAD,\nDAMMAPETA(V),DAMMAPETA(M),BHADRADRI(D),TG,IND\n36-TELANGANA\nPHONE. : 9866514042\nD.L.No. : TS/KGM/2023-106432', None, None, None, None, None, None]
[None, None, None, None, None, None, None, 'Invoice No', None, None, 'B001000786', None, None, None, 'Order No.\nOrder Date', None, None, None, 'Cases 0', None, None, None, None, None, None, None, None, None]
[None, None, None, None, None, None, None, 'Invoice Date\nDue Date', None, None, '13-07-2023\n03-08-2023', None, None, None, 'L.R. No.\nL.R. Date13-07-2023', None, None, None,

stage_2 

In [5]:
import json

def ollama_generate(prompt, model=GEN_MODEL, temperature=0.0, json_mode=True):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_ctx": 4096},
    }
    if json_mode:
        payload["format"] = "json"
    r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=600)
    r.raise_for_status()
    return r.json()["response"]

print(ollama_generate('reply with json:{"ok":true}'))

{"ok": true}


In [6]:
EXTRACTION_SCHEMA = {
    "seller": {
        "name": "", "address": "", "phone": "",
        "gstin": "", "drug_license_no": ""
    },
    "buyer": {
        "name": "", "address": "", "phone": "",
        "state_code": "", "drug_license_no": ""
    },
    "invoice": {
        "number": "", "date": "", "due_date": "",
        "type": "",          # CREDIT / CASH
        "lr_no": "", "lr_date": "",
        "order_no": "", "order_date": "",
        "transport": "", "cases": 0
    },
    "line_items": [
        {
            "sl_no": 0, "hsn": "", "mfr": "", "product_name": "",
            "pack": "", "batch": "", "expiry": "",
            "qty": 0.0, "free": 0.0,
            "mrp": 0.0, "rate": 0.0, "discount_pct": 0.0,
            "sgst_pct": 0.0, "cgst_pct": 0.0,
            "amount": 0.0, "net_amount": 0.0
        }
    ],
    "tax_summary": {
        "by_slab": [
            {"slab_pct": 0.0, "taxable": 0.0, "sgst": 0.0, "cgst": 0.0, "total_gst": 0.0}
        ],
        "total_taxable": 0.0,
        "total_sgst": 0.0,
        "total_cgst": 0.0,
        "total_gst": 0.0
    },
    "totals": {
        "items_count": 0,
        "qty_total": 0.0,
        "discount_total": 0.0,
        "freight": 0.0,
        "cr_dr_note": 0.0,
        "grand_total": 0.0,
        "grand_total_in_words": ""
    }
}

EXTRACTION_PROMPT = """You are extracting fields from a GST invoice.
Return ONE JSON object exactly matching this schema. Use null for missing values.
Numbers must be numbers, not strings. Dates as YYYY-MM-DD when possible.

SCHEMA:
{schema}

INVOICE TEXT:
{text}

EXTRACTED TABLES (raw rows from PDF):
{tables}

Return ONLY the JSON, no commentary."""

In [ ]:
def extract_fields(text, tables):
    prompt = EXTRACTION_PROMPT.format(
        schema=json.dumps(EXTRACTION_SCHEMA, indent=2),
        text=text[:2000],
        tables=json.dumps([t["rows"] for t in tables], indent=2)[:1500],
    )
    raw = ollama_generate(prompt)
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"[!] JSON parse failed: {e}\nRaw output:\n{raw[:500]}")
        return {}

extracted = extract_fields(text, tables)
print(json.dumps(extracted, indent=2)[:3000])